# 🏠 House Price Prediction with Neural Networks 

This notebook demonstrates the **fundamental workflow** to build, train, and evaluate a neural network in **PyTorch** for a regression task: predicting house prices.

## What we'll learn:
- **Data loading and preprocessing** from Kaggle
- **Exploratory data analysis** to understand the features
- **Building a neural network** with PyTorch (BatchNorm, Dropout)
- **Training with early stopping** to prevent overfitting
- **Evaluating model performance** on unseen test data
- **Visualizing results** and understanding predictions

---

## 📦 Step 0: Install Required Packages

Before we begin, we need to install the tools (libraries) that our code depends on:

| Library | Purpose |
|---------|----------|
| `torch` | PyTorch — the deep learning framework we use to build and train the neural network |
| `scikit-learn` | Provides utilities for data splitting and scaling |
| `kagglehub` | Downloads datasets directly from Kaggle |
| `pandas` | Works with tabular data (like an Excel spreadsheet) |
| `numpy` | Fast numerical operations on arrays |
| `matplotlib` / `seaborn` | Creating static charts and visualizations |

**Run this cell once** — it may take a minute to install everything.

In [ ]:
# ============================================
# INSTALLATION — Run this first!
# ============================================

%pip install torch scikit-learn kagglehub pandas numpy matplotlib seaborn --quiet

print("✅ All packages installed successfully!")

## 📦 Step 1: Import Libraries

Installing a package puts it on your computer; **importing** it makes it available in your code.

Think of it as: *installing = buying the toolbox, importing = opening the toolbox on your workbench.*

In [ ]:
# ============================================
# IMPORT LIBRARIES
# ============================================

# --- Deep learning ---
import torch                             # Core PyTorch library
import torch.nn as nn                    # Neural network building blocks

# --- Data handling ---
import numpy as np                       # Numerical arrays and math
import pandas as pd                      # DataFrames (tabular data)
import kagglehub                         # Download datasets from Kaggle
import os                                # File-system utilities

# --- Machine learning utilities ---
from sklearn.model_selection import train_test_split  # Split data
from sklearn.preprocessing import StandardScaler      # Scale data

# --- Visualization ---
import matplotlib.pyplot as plt          # Static plots
import seaborn as sns                    # Statistical plots

# Apply a clean visual style for all plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("✅ All libraries imported successfully!")

---

# 📊 1. Data Loading & Exploration

## About the Dataset

We use a **synthetic (computer-generated) house price dataset** from Kaggle containing **10,000 houses**.

Each house is described by these features:

| Feature | Description |
|---------|------------|
| `square_feet` | Size of the house in square feet |
| `num_rooms` | Number of rooms |
| `age` | Age of the house in years |
| `distance_to_city(km)` | Distance from city center in km |
| `price` | The house price — **this is what we want to predict!** |

The code below **automatically downloads** the dataset from Kaggle using `kagglehub`.

In [ ]:
# ============================================
# DOWNLOAD DATASET FROM KAGGLE
# ============================================
# kagglehub.dataset_download() fetches the dataset
# and caches it locally so it won't re-download next time.

print("📥 Downloading dataset from Kaggle...")
dataset_path = kagglehub.dataset_download(
    'muhamedumarjamil/house-price-prediction-dataset'
)
print(f"✅ Dataset downloaded to: {dataset_path}")

### Loading the CSV into a DataFrame

A **DataFrame** (from the `pandas` library) is essentially a table with labelled columns — very similar to a spreadsheet. We use `pd.read_csv()` to read the CSV file that was just downloaded.

In [ ]:
# ============================================
# LOAD CSV INTO A PANDAS DATAFRAME
# ============================================

csv_file = os.path.join(dataset_path, 'house_prices_dataset.csv')
df = pd.read_csv(csv_file)

print(f"✅ Dataset loaded!")
print(f"   Rows (houses):    {len(df)}")
print(f"   Columns (features): {len(df.columns)}")
print(f"   Column names: {list(df.columns)}")

### 🧹 Data Cleaning

Before we proceed, we need to clean the data by removing **extreme outliers** — we use log-space to identify and remove houses with prices more than 3 standard deviations from the mean.

In [ ]:
# ============================================
# DATA CLEANING
# ============================================

print(f"📊 Original dataset size: {len(df)} samples")

# Remove extreme outliers using log-space (±3 std)
# This helps the model learn better by removing houses with unusual prices
log_prices = np.log(df['price'])
mean_log = log_prices.mean()
std_log = log_prices.std()
outlier_mask = np.abs(log_prices - mean_log) <= 3 * std_log
df = df[outlier_mask].copy()

print(f"   After removing outliers: {len(df)} samples")
print(f"✅ Data cleaning complete!")

### Visual exploration

Before building a model, it is always a good idea to **visualize** the data. We create four plots:

1. **Price distribution** — is it symmetric? skewed? are there outliers?
2. **Feature correlations** — which features are most/least related to price?
3. **Size vs Price** scatter — the single strongest predictor.
4. **Age vs Price** scatter — does age make a house cheaper or more expensive?

In [ ]:
# ============================================
# DATA VISUALIZATION
# ============================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- 1. Distribution of house prices ---
axes[0, 0].hist(df['price'], bins=50, color='steelblue',
                edgecolor='white', alpha=0.7)
axes[0, 0].set_xlabel('House Price ($)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('📊 Distribution of House Prices')
axes[0, 0].axvline(df['price'].mean(), color='red', linestyle='--',
                    label=f'Mean: ${df["price"].mean():,.0f}')
axes[0, 0].legend()

# --- 2. Feature correlation with price ---
feature_cols = [c for c in df.columns if c != 'price']
correlations = [df[c].corr(df['price']) for c in feature_cols]
colors = ['green' if c > 0 else 'red' for c in correlations]
axes[0, 1].barh(feature_cols, correlations, color=colors, alpha=0.7)
axes[0, 1].set_xlabel('Correlation with Price')
axes[0, 1].set_title('📈 Feature Correlation with House Price')
axes[0, 1].axvline(0, color='black', linewidth=0.5)

# --- 3. Square Feet vs Price ---
axes[1, 0].scatter(df['square_feet'], df['price'], alpha=0.3, s=5,
                   c='steelblue')
axes[1, 0].set_xlabel('Square Feet')
axes[1, 0].set_ylabel('Price ($)')
axes[1, 0].set_title('📐 House Size vs Price')

# --- 4. Age vs Price ---
axes[1, 1].scatter(df['age'], df['price'], alpha=0.3, s=5,
                   c='green')
axes[1, 1].set_xlabel('Age (years)')
axes[1, 1].set_ylabel('Price ($)')
axes[1, 1].set_title('🕰️ House Age vs Price')

plt.tight_layout()
plt.show()

---

# 🔧 2. Data Preparation

Raw data cannot be fed directly into a neural network — we need preparation steps:

1. **Separate features and target** — the model receives the features as input and tries to predict the target (price).
2. **Split** the data into training, validation, and test sets.
3. **Scale** (standardize) the values so all features live on a similar numeric range.
4. **Convert** to PyTorch tensors.

## Why three sets?

| Set | % of data | Role |
|-----|-----------|------|
| Training | 80% | The model **learns** from this data |
| Validation | 10% | Used to **monitor** training and detect overfitting |
| Test | 10% | **Final evaluation** on data the model has never seen |

### 2a. Separate features and target

We pull the **input features** (square_feet, num_rooms, age, distance_to_city) into one array and the **target** (price) into another.

In this base approach we **predict the raw price in dollars** — no transformations on the target. This is the simplest possible setup: the model outputs a number, and we compare it directly against the actual price.

In [ ]:
# ============================================
# SEPARATE FEATURES AND TARGET
# ============================================

# Feature columns — everything except 'price'
feature_columns = [c for c in df.columns if c != 'price']

# .values converts a DataFrame column into a NumPy array
features = df[feature_columns].values          # shape: (N, 4)
prices = df['price'].values.reshape(-1, 1)     # shape: (N, 1)

print(f"Number of samples:  {len(prices)}")
print(f"Number of features: {features.shape[1]}")
print(f"Feature names:      {feature_columns}")
print(f"\n📊 Price statistics:")
print(f"   Range: ${prices.min():,.0f} - ${prices.max():,.0f}")
print(f"   Mean:  ${prices.mean():,.0f}")

### 2b. Split the data

`train_test_split` from scikit-learn randomly shuffles the data and divides it. We call it **twice** to create three sets from the original data:

```
Full data  ──▶  80% Training  +  20% Temp
         Temp  ──▶  50% Validation  +  50% Test
```

This gives us an **80/10/10** split (training / validation / test).

Setting `random_state=42` makes the split **reproducible** — you get the same split every time you run the cell.

In [ ]:
# ============================================
# SPLIT THE DATA INTO TRAIN / VAL / TEST
# ============================================

# First split: 80% training, 20% temporary
train_features, temp_features, train_prices, temp_prices = train_test_split(
    features, prices, test_size=0.2, random_state=42
)

# Second split: 50/50 on the temporary set → 10% val + 10% test
val_features, test_features, val_prices, test_prices = train_test_split(
    temp_features, temp_prices, test_size=0.5, random_state=42
)

print(f"Training samples:   {len(train_prices)}")
print(f"Validation samples: {len(val_prices)}")
print(f"Test samples:       {len(test_prices)}")

### 2c. Scale (standardize) the features

Neural networks learn much faster when all input values are on a **similar scale**.

**Standardization** transforms every feature column so that it has:
- **mean = 0**
- **standard deviation = 1**

The formula for each value $x$ is:

$$x_{\text{scaled}} = \frac{x - \mu}{\sigma}$$

where $\mu$ is the column mean and $\sigma$ is the standard deviation.

> ⚠️ **Important:** we call `.fit_transform()` on the **training set only**, then `.transform()` on validation and test sets. This prevents *data leakage* — the model must not see statistics from data it will be evaluated on.

> 💡 **Note:** We scale both features and the target (prices). Since we predict raw dollar values, the model benefits from having the target in a standardized range too. We'll need the price scaler later to convert predictions back to real dollars.

In [ ]:
# ============================================
# SCALE THE DATA
# ============================================

# Scaler for features
feature_scaler = StandardScaler()
train_features = feature_scaler.fit_transform(train_features)
val_features   = feature_scaler.transform(val_features)
test_features  = feature_scaler.transform(test_features)

# Scaler for the target (prices in dollars)
# We need this to convert predictions back to real prices later
price_scaler = StandardScaler()
train_prices_scaled = price_scaler.fit_transform(train_prices)
val_prices_scaled   = price_scaler.transform(val_prices)
test_prices_scaled  = price_scaler.transform(test_prices)

print("✅ Features and prices scaled — all values now have mean ≈ 0 and std ≈ 1.")
print(f"   Price mean used for scaling: ${price_scaler.mean_[0]:,.0f}")
print(f"   Price std used for scaling:  ${price_scaler.scale_[0]:,.0f}")

### 2d. Convert to PyTorch tensors

PyTorch works with its own data type called a **tensor**, which is similar to a NumPy array but can also run on a GPU for faster computation. We convert all our arrays now.

In [ ]:
# ============================================
# CONVERT TO PYTORCH TENSORS
# ============================================

# Convert features to tensors
train_features_t = torch.tensor(train_features, dtype=torch.float32)
val_features_t   = torch.tensor(val_features,   dtype=torch.float32)
test_features_t  = torch.tensor(test_features,  dtype=torch.float32)

# Convert scaled prices to tensors (these are what we train on)
train_prices_t = torch.tensor(train_prices_scaled, dtype=torch.float32)
val_prices_t   = torch.tensor(val_prices_scaled,   dtype=torch.float32)
test_prices_t  = torch.tensor(test_prices_scaled,  dtype=torch.float32)

print(f"Training tensor shapes:   features {train_features_t.shape}, prices {train_prices_t.shape}")
print(f"Validation tensor shapes: features {val_features_t.shape}, prices {val_prices_t.shape}")
print(f"Test tensor shapes:       features {test_features_t.shape}, prices {test_prices_t.shape}")

---

# 🧠 3. Neural Network Architecture

## What is a neural network?

A neural network is a series of **layers** of interconnected "neurons". Each layer takes numbers in, applies a weighted sum plus a non-linear function, and passes the result to the next layer. By adjusting the weights during training, the network learns to map inputs to the desired output.

## Our architecture

We use a **4-layer fully-connected network** with BatchNorm and Dropout:

```
Input (4 features) → 128 → 128 → 64 → 1 (scaled price)
```

```
  ┌───────────┐     ┌───────────┐     ┌───────────┐     ┌───────────┐     ┌──────────┐
  │  INPUT    │     │  LAYER 1  │     │  LAYER 2  │     │  LAYER 3  │     │  OUTPUT  │
  │ (4 feat.) │────▶│128 neurons│────▶│128 neurons│────▶│ 64 neurons│────▶│ 1 neuron │
  │           │     │           │     │           │     │           │     │          │
  │sq_feet    │     │ Linear    │     │ Linear    │     │ Linear    │     │ Linear   │
  │num_rooms  │     │ BatchNorm │     │ BatchNorm │     │ BatchNorm │     │ (no act.)│
  │age        │     │ ReLU      │     │ ReLU      │     │ ReLU      │     │          │
  │distance   │     │ Dropout   │     │ Dropout   │     │ Dropout/2 │     │          │
  └───────────┘     └───────────┘     └───────────┘     └───────────┘     └──────────┘
```

### Key components

| Component | What it does |
|-----------|-------------|
| `nn.Linear(in, out)` | A fully-connected layer: computes `out = weight × in + bias` |
| `nn.BatchNorm1d(n)` | Normalizes layer outputs for more stable training |
| `nn.ReLU()` | Activation function: keeps positive values, sets negatives to 0 |
| `nn.Dropout(p)` | During training, randomly sets a fraction `p` of neurons to 0 — this prevents **overfitting** |

### Why BatchNorm?

BatchNorm normalizes the inputs to each layer, which helps with:
- **Faster convergence** — allows using higher learning rates
- **Regularization** — adds slight noise during training
- **Stable gradients** — reduces internal covariate shift

### Why Dropout?

Dropout randomly "disables" neurons during training. This forces the network to not rely on any single neuron, leading to more **robust** learned features. At evaluation time, dropout is turned off and all neurons are active.

In [ ]:
# ============================================
# DEFINE THE NEURAL NETWORK MODEL
# ============================================

class HousePriceModel(nn.Module):
    """
    A neural network for house price prediction.

    Architecture:
        Input → 128 → 128 → 64 → 1
        with BatchNorm, ReLU activations, and Dropout regularization.
    """

    def __init__(self, n_features, dropout_rate=0.2):
        super().__init__()

        self.network = nn.Sequential(
            # Layer 1: input features → 128 neurons
            nn.Linear(n_features, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            # Layer 2: 128 → 128 neurons
            nn.Linear(128, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            # Layer 3: 128 → 64 neurons
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout_rate / 2),  # Reduced dropout near output

            # Output layer: 64 → 1 (predicted scaled price)
            nn.Linear(64, 1)
        )

    def forward(self, x):
        """Forward pass: push input x through all layers."""
        return self.network(x)


# Instantiate the model
model = HousePriceModel(
    n_features=train_features_t.shape[1],  # 4 features
    dropout_rate=0.2
)

print(model)
print(f"\n📊 Total parameters: {sum(p.numel() for p in model.parameters()):,}")

---

# 🏋️ 4. Training the Model

Training means repeatedly:

1. **Forward pass** — the model makes predictions for all training houses.
2. **Loss computation** — we measure how wrong the predictions are (Mean Squared Error).
3. **Backward pass** — PyTorch calculates the gradient of the loss with respect to every weight.
4. **Optimizer step** — the optimizer adjusts the weights to reduce the loss.

One full cycle through the training set is called an **epoch**.

## Hyperparameters

| Hyperparameter | Value | Meaning |
|---------------|-------|---------| 
| `learning_rate` | 0.001 | How big each weight update step is |
| `weight_decay` | 1e-4 | L2 regularization — penalizes large weights |
| `dropout_rate` | 0.2 | Fraction of neurons randomly disabled each step |
| `max_epochs` | 2000 | Maximum training loops (early stopping will likely stop earlier) |

## Training strategy

In this base approach we use **full-batch training**: at each epoch, the model sees the **entire** training set at once. This is the simplest approach — we'll see an alternative (mini-batches) in the next notebook.

We also use **early stopping**: if the validation loss doesn't improve for 100 consecutive epochs, we stop training and restore the best model weights.

In [ ]:
# ============================================
# SET UP LOSS FUNCTION AND OPTIMIZER
# ============================================

# MSELoss = Mean Squared Error: average of (prediction - actual)²
loss_function = nn.MSELoss()

# Learning rate — controls step size during optimization
learning_rate = 0.001

# Weight decay — L2 regularization strength
weight_decay = 1e-4

# Adam optimizer — a popular gradient-descent variant that adapts
# the learning rate for each parameter individually.
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay
)

# Number of full passes over the training data
number_of_epochs = 2000

print(f"Loss function: {loss_function}")
print(f"Optimizer:     Adam (lr={learning_rate}, weight_decay={weight_decay})")
print(f"Max epochs:    {number_of_epochs}")

### The training loop

Inside each epoch we do **two things**:

1. **Train** — `model.train()` activates dropout and BatchNorm training mode. We compute the loss, back-propagate gradients, and update weights.
2. **Validate** — `model.eval()` disables dropout and switches BatchNorm to eval mode. We compute the validation loss (no weight updates) to monitor generalization.

We also implement **early stopping**: we track the best validation loss and if it doesn't improve for `patience` epochs, we stop and restore the best weights.

In [ ]:
# ============================================
# TRAINING LOOP (with Early Stopping)
# ============================================

# Lists to store loss values for later plotting
train_losses = []
val_losses   = []

# Early stopping setup
best_val_loss = float('inf')
patience_counter = 0
patience = 100  # Stop if no improvement for 100 epochs
best_model_state = None

for epoch in range(number_of_epochs):

    # --- TRAINING PHASE ---
    model.train()                                        # Enable dropout & BatchNorm training mode
    predicted_prices = model(train_features_t)           # Forward pass
    train_loss = loss_function(predicted_prices,
                               train_prices_t)           # Compute loss

    optimizer.zero_grad()           # Clear old gradients
    train_loss.backward()           # Compute new gradients (backpropagation)
    optimizer.step()                # Update weights

    # --- VALIDATION PHASE ---
    model.eval()                                         # Disable dropout, BatchNorm in eval mode
    with torch.no_grad():                                # No gradient tracking
        val_predictions = model(val_features_t)
        val_loss = loss_function(val_predictions,
                                 val_prices_t)

    # Store losses
    train_losses.append(train_loss.item())
    val_losses.append(val_loss.item())

    # --- EARLY STOPPING CHECK ---
    if val_loss.item() < best_val_loss:
        best_val_loss = val_loss.item()
        patience_counter = 0
        # Save the best model state
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print(f"\n⏹️  Early stopping triggered at epoch {epoch}")
        print(f"   No improvement for {patience} epochs")
        break

    # Print progress every 100 epochs
    if epoch % 100 == 0:
        print(
            f"Epoch {epoch:>4d} | "
            f"Train: {train_loss.item():.4f} | "
            f"Val: {val_loss.item():.4f}"
        )

# Load the best model state
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"\n✅ Loaded best model from epoch with val_loss = {best_val_loss:.4f}")

print(f"\n✅ Training complete!")
print(f"   Total epochs run:      {epoch + 1}")
print(f"   Best validation loss:  {best_val_loss:.4f}")

### Visualizing the training history

Plotting the training and validation loss over epochs tells us:

- **Are we learning?** — Both curves should go down.
- **Are we overfitting?** — If validation loss starts rising while training loss keeps falling, the model is memorizing training data.
- **Did we train long enough?** — If the curves are still dropping, more epochs might help.

In [ ]:
# ============================================
# TRAINING HISTORY VISUALIZATION
# ============================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: full loss curves (log scale for better visibility)
axes[0].plot(train_losses, label='Training Loss', alpha=0.8)
axes[0].plot(val_losses,   label='Validation Loss', alpha=0.8)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss (log scale)')
axes[0].set_title('📉 Training & Validation Loss Over Time')
axes[0].legend()
axes[0].set_yscale('log')

# Right: zoomed view of last portion of training
zoom_epochs = min(200, len(train_losses))
axes[1].plot(range(len(train_losses) - zoom_epochs, len(train_losses)),
             train_losses[-zoom_epochs:], label='Training Loss', alpha=0.8)
axes[1].plot(range(len(val_losses) - zoom_epochs, len(val_losses)),
             val_losses[-zoom_epochs:], label='Validation Loss', alpha=0.8)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MSE Loss')
axes[1].set_title(f'🔍 Loss Convergence (Last {zoom_epochs} Epochs)')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\n✅ Final Training Loss:   {train_losses[-1]:.4f}")
print(f"✅ Final Validation Loss: {val_losses[-1]:.4f}")

---

# 📏 5. Model Evaluation

Now for the real test: how well does the model predict prices for houses **it has never seen**?

We evaluate on the **test set** — data that was held out from both training and validation.

## Metrics we compute

**MAE (Mean Absolute Error):** — Average absolute error in dollars.

**RMSE (Root Mean Squared Error):**  — Penalizes large errors more heavily.

**MAPE (Mean Absolute Percentage Error):**  — Error as a percentage of the actual price.



In [ ]:
# ============================================
# GENERATE PREDICTIONS AND CONVERT BACK TO DOLLARS
# ============================================

model.eval()
with torch.no_grad():
    # Get predictions in scaled space
    test_pred_scaled = model(test_features_t).numpy()

# Convert from scaled space back to real prices
test_predictions = price_scaler.inverse_transform(test_pred_scaled)

# Original test prices (already in dollars)
test_prices_original = test_prices

In [ ]:
# ============================================
# OVERALL PERFORMANCE METRICS
# ============================================

# Flatten arrays for metrics calculation
actual = test_prices_original.flatten()
pred = test_predictions.flatten()

# Calculate overall metrics
mae = np.mean(np.abs(pred - actual))
rmse = np.sqrt(np.mean((pred - actual) ** 2))
mape = np.mean(np.abs((pred - actual) / actual)) * 100



print("=" * 55)
print("           OVERALL TEST SET PERFORMANCE")
print("=" * 55)
print()
print(f"   MAE  (Mean Absolute Error):      ${mae:,.2f}")
print(f"   RMSE (Root Mean Squared Error):  ${rmse:,.2f}")
print(f"   MAPE (Mean Absolute % Error):    {mape:.2f}%")
print()
print("=" * 55)

In [ ]:
# ============================================
# PERFORMANCE BY PRICE RANGE
# ============================================

# Define quartile boundaries
quartiles = np.quantile(actual, [0, 0.25, 0.5, 0.75, 1.0])

print("\n" + "=" * 55)
print("        PERFORMANCE BY PRICE RANGE (QUARTILES)")
print("=" * 55)

for i in range(4):
    # Select samples in this price range
    mask = (actual >= quartiles[i]) & (actual < quartiles[i+1])
    a = actual[mask]
    p = pred[mask]

    # Metrics
    mae_q  = np.mean(np.abs(p - a))
    rmse_q = np.sqrt(np.mean((p - a) ** 2))
    mape_q = np.mean(np.abs((p - a) / a)) * 100

    print()
    print(f"Range {i+1}: ${quartiles[i]:,.0f} → ${quartiles[i+1]:,.0f}")
    print(f"   MAE:  ${mae_q:,.2f}")
    print(f"   RMSE: ${rmse_q:,.2f}")
    print(f"   MAPE: {mape_q:.2f}%")

---

# 📊 6. Prediction Visualizations

Numbers alone don't always tell the full story. Let's create three complementary visualizations:

1. **Predicted vs Actual** scatter plot — ideally, all points should lie on the diagonal.
2. **Residuals histogram** — the distribution of errors; a narrow, centered bell-shape is ideal.
3. **Sample comparison** — a bar chart comparing actual and predicted prices for 20 random houses.

In [ ]:
# ============================================
# PREDICTION VISUALIZATIONS
# ============================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- 1. Predicted vs Actual scatter ---
axes[0].scatter(test_prices_original, test_predictions,
                alpha=0.5, s=10)
lo = min(test_prices_original.min(), test_predictions.min())
hi = max(test_prices_original.max(), test_predictions.max())
axes[0].plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='Perfect')
axes[0].set_xlabel('Actual Price ($)')
axes[0].set_ylabel('Predicted Price ($)')
axes[0].set_title('🎯 Predicted vs Actual Prices')
axes[0].legend()

# --- 2. Residuals distribution ---
residuals = (test_predictions - test_prices_original).flatten()
axes[1].hist(residuals, bins=50, color='steelblue',
             edgecolor='white', alpha=0.7)
axes[1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Prediction Error ($)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('📊 Distribution of Prediction Errors')

# --- 3. Sample comparison bar chart ---
np.random.seed(42)
sample_idx = np.random.choice(len(test_prices_original), 20, replace=False)
x_pos = np.arange(20)
width = 0.35
axes[2].bar(x_pos - width / 2,
            test_prices_original[sample_idx].flatten(),
            width, label='Actual', alpha=0.8)
axes[2].bar(x_pos + width / 2,
            test_predictions[sample_idx].flatten(),
            width, label='Predicted', alpha=0.8)
axes[2].set_xlabel('Sample Index')
axes[2].set_ylabel('Price ($)')
axes[2].set_title('🏠 Sample Predictions vs Actual')
axes[2].legend()
axes[2].set_xticks(x_pos)

plt.tight_layout()
plt.show()

---

## 🎯 Summary

In this notebook we built a **complete neural network pipeline** from scratch:

1. ✅ Loaded and explored the data
2. ✅ Preprocessed features with standardization
3. ✅ Built a 4-layer network with BatchNorm and Dropout
4. ✅ Trained with early stopping
5. ✅ Evaluated on held-out test data
6. ✅ Visualized predictions

**In the next notebook**, we'll apply some specific tricks to improve this model's performance. Stay tuned!